In [ ]:
# ======================================================
# Cell 0: Environment Setup — Run this first!
# ======================================================
# Detects Google Colab vs local Jupyter and sets three variables used
# throughout the notebook:
#
#   DATA_FILE     — absolute path to Data/mapped-dataset-36-20.xlsx
#   OUTPUT_DIR    — directory where results/reports are saved
#   BASELINE_FILE — path to summary_statistics_RTSGA_only_RTW_10%.xlsx
#
# ── Google Colab users ──────────────────────────────────────────────
#   Upload the project to Google Drive, then set DRIVE_PROJECT_PATH
#   below to the folder that contains the Data/ subfolder.
#
# ── Local Jupyter users ─────────────────────────────────────────────
#   No action needed. Paths are found automatically by walking up from
#   the current working directory until Data/mapped-dataset-36-20.xlsx
#   is found.
# ======================================================

import subprocess, sys, os
from pathlib import Path

# ── Install dependencies ──────────────────────────────────────────────
for _pkg in ["pandas", "numpy", "matplotlib", "seaborn", "scipy", "openpyxl", "ipywidgets"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", _pkg, "-q"])
print("✅ All dependencies installed.")

# ── Detect environment ────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Resolve paths ─────────────────────────────────────────────────────
_DATA_FILENAME = "mapped-dataset-36-20.xlsx"

if IN_COLAB:
    # ─── COLAB — Option A: Google Drive (recommended) ─────────────────
    # Set DRIVE_PROJECT_PATH to the Drive folder that contains Data/ and RTSGA/
    DRIVE_PROJECT_PATH = "/content/drive/MyDrive/BV-Driven-History-RTSGA"  # ← adjust if needed

    try:
        from google.colab import drive
        drive.mount("/content/drive")
        _data_candidate = Path(DRIVE_PROJECT_PATH) / "Data" / _DATA_FILENAME
        if not _data_candidate.exists():
            raise FileNotFoundError(
                f"{_DATA_FILENAME} not found at {_data_candidate}. "
                "Check DRIVE_PROJECT_PATH and make sure you uploaded the project to Drive."
            )
        DATA_FILE     = str(_data_candidate)
        OUTPUT_DIR    = str(Path(DRIVE_PROJECT_PATH) / "RTSGA")
        BASELINE_FILE = str(Path(OUTPUT_DIR) / "summary_statistics_RTSGA_only_RTW_10%.xlsx")

    except Exception as _e:
        # ─── COLAB — Option B: File upload fallback ────────────────────
        print(f"Drive mount failed ({_e}).\nFalling back to manual file upload.")
        from google.colab import files as _cf

        print(f"\nStep 1 — Upload {_DATA_FILENAME}:")
        _up = _cf.upload()
        DATA_FILE = next(iter(_up))

        OUTPUT_DIR = "/content"

        print("\nStep 2 — Upload summary_statistics_RTSGA_only_RTW_10%.xlsx (Cancel to skip):")
        try:
            _up2 = _cf.upload()
            BASELINE_FILE = next(iter(_up2)) if _up2 else None
        except Exception:
            BASELINE_FILE = None

else:
    # ─── LOCAL — walk up from CWD to find the project root ────────────
    def _find_root(start: Path, marker: str):
        for p in [start] + list(start.parents):
            if (p / marker).exists():
                return p
        return None

    _root = _find_root(Path.cwd(), os.path.join("Data", _DATA_FILENAME))
    if _root is None:
        raise FileNotFoundError(
            f"Could not find Data/{_DATA_FILENAME} by walking up from the current directory.\n"
            "Open this notebook from inside the project folder, or set DATA_FILE manually."
        )
    DATA_FILE     = str(_root / "Data" / _DATA_FILENAME)
    OUTPUT_DIR    = str(Path.cwd())          # outputs saved next to the notebook
    BASELINE_FILE = str(Path.cwd() / "summary_statistics_RTSGA_only_RTW_10%.xlsx")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nEnvironment  : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"Data file    : {DATA_FILE}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Baseline file: {BASELINE_FILE}")

In [ ]:
# ===============================
# Cell 1: Setup and Configuration
# ===============================
import pandas as pd
import numpy as np
import random
import time
import os
import sys
import subprocess
import re
import ast
import ipywidgets as widgets
from IPython.display import display
import io
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import Workbook
from openpyxl.drawing.image import Image as OpenpyxlImage
from io import BytesIO

# --- CONFIGURATION FOR THE EXPERIMENT ---
BUDGET_PERCENTAGES = [10]
NUM_RUNS_PER_BUDGET = 30

GA_CONFIGURATIONS = {
    "V5_AG": {
        "max_generations": 100, "crossover_prob": 0.08, "mutation_prob": 0.05,
    }
}

In [3]:
# ======================================================
# Cell 2: The GA Logic as a Reusable Function
# ======================================================

def run_ga_instance(max_exec_time, ga_params, data_maps):
    # Unpack GA parameters and data maps
    max_generations = ga_params['max_generations']
    crossover_prob = ga_params['crossover_prob']
    mutation_prob = ga_params['mutation_prob']

    req_to_tests = data_maps['req_to_tests']
    req_to_bv = data_maps['req_to_bv']
    test_to_time = data_maps['test_to_time']
    all_req_ids = data_maps['all_req_ids']
    bval_to_reqs_map = data_maps['bval_to_reqs_map']
    n_requirements = len(all_req_ids)

    # ----------------------------------------------------------------------
    # Helper: Evaluation Decomposition (Deb, 2000)
    # Source: Deb (2000), "An Efficient Constraint Handling Method for GAs"
    # ----------------------------------------------------------------------
    def evaluate_chrom(chrom):
        covered_tests = set(); total_bv = 0
        for idx, val in enumerate(chrom):
            if val:
                req = all_req_ids[idx]
                covered_tests |= req_to_tests.get(req, set())
                total_bv += req_to_bv.get(req, 0)
        total_time = sum(test_to_time.get(t, 0) for t in covered_tests)
        feasible = (total_time <= max_exec_time)
        violation = max(0, total_time - max_exec_time)
        return total_bv, total_time, feasible, violation

    # ----------------------------------------------------------------------
    # Feasible Initialization (Chu & Beasley, 1998)
    # ----------------------------------------------------------------------
    def feasible_chromosome():
        chrom = [0] * n_requirements; indices = list(range(n_requirements)); random.shuffle(indices)
        covered_tests = set(); total_time = 0
        for idx in indices:
            req = all_req_ids[idx]; tests = req_to_tests.get(req, set())
            new_tests = tests - covered_tests
            additional_time = sum(test_to_time.get(t, 0) for t in new_tests)
            if total_time + additional_time <= max_exec_time:
                chrom[idx] = 1; covered_tests |= new_tests; total_time += additional_time
        return chrom

    population_size = 10
    population = [feasible_chromosome() for _ in range(population_size)]

    # ----------------------------------------------------------------------
    # Greedy Seeding for Budgeted BV Coverage (Khuller, Moss, Naor, 1999)
    # Source: Khuller–Moss–Naor (1999), Budgeted Maximum Coverage (1−1/e)
    # ----------------------------------------------------------------------
    def greedy_seed_variant():
        chrom = [0] * n_requirements; covered = set(); used_time = 0
        remaining = set(range(n_requirements))
        while remaining:
            best_idx, best_score = None, float("-inf")
            for idx in remaining:
                req = all_req_ids[idx]; tests = req_to_tests.get(req, set()); new_tests = tests - covered
                add_time = sum(test_to_time.get(t, 0) for t in new_tests)
                if add_time < 0: continue
                score = req_to_bv.get(req, 0) if add_time == 0 else (req_to_bv.get(req, 0) / (add_time + 1e-12))
                if used_time + add_time <= max_exec_time and score > best_score:
                    best_idx, best_score = idx, score
            if best_idx is None: break
            req = all_req_ids[best_idx]; new_tests = req_to_tests.get(req, set()) - covered
            add_time = sum(test_to_time.get(t, 0) for t in new_tests)
            if used_time + add_time <= max_exec_time:
                chrom[best_idx] = 1; covered |= new_tests; used_time += add_time
            remaining.remove(best_idx)
        return chrom

    # Inject a few seeds to improve the starting population
    num_seeds = min(3, population_size)
    for i in range(num_seeds):
        population[i] = greedy_seed_variant()

    # ----------------------------------------------------------------------
    # Value-Aware Repair / Improve Operator
    # Source: Chu & Beasley (1998); randomized/efficiency-based repair literature
    # ----------------------------------------------------------------------
    def repair_value_aware(chrom):
        chrom = chrom[:]
        # DROP phase: remove the least efficient requirement until feasible
        while True:
            _, _, feasible, _ = evaluate_chrom(chrom)
            if feasible: break
            selected = [i for i, v in enumerate(chrom) if v]
            if not selected: break
            worst_idx, worst_ratio = None, float("inf")
            for idx in selected:
                req = all_req_ids[idx]; covered_other = set()
                for j, v in enumerate(chrom):
                    if v and j != idx: covered_other |= req_to_tests.get(all_req_ids[j], set())
                unique_tests = req_to_tests.get(req, set()) - covered_other
                time_save = sum(test_to_time.get(t, 0) for t in unique_tests)
                bv_loss = req_to_bv.get(req, 0)
                ratio = bv_loss / (time_save + 1e-12) if time_save > 0 else float("inf")
                ratio += random.uniform(-1e-9, 1e-9)
                if ratio < worst_ratio: worst_ratio, worst_idx = ratio, idx
            if worst_idx is None: break
            chrom[worst_idx] = 0
        # ADD phase: greedily add the most efficient requirement that fits
        while True:
            _, tt, _, _ = evaluate_chrom(chrom); slack = max_exec_time - tt
            if slack <= 0: break
            candidates = [i for i, v in enumerate(chrom) if not v]; best_idx, best_score = None, float("-inf")
            covered_now = set()
            for j, v in enumerate(chrom):
                if v: covered_now |= req_to_tests.get(all_req_ids[j], set())
            for idx in candidates:
                req = all_req_ids[idx]; new_tests = req_to_tests.get(req, set()) - covered_now
                add_time = sum(test_to_time.get(t, 0) for t in new_tests)
                if add_time < 0: continue
                score = req_to_bv.get(req, 0) if add_time == 0 else (req_to_bv.get(req, 0) / (add_time + 1e-12))
                score += random.uniform(-1e-9, 1e-9)
                if add_time <= slack and score > best_score: best_idx, best_score = idx, score
            if best_idx is None: break
            chrom[best_idx] = 1
        return chrom

    # ----------------------------------------------------------------------
    # Fitness Function (Yoo & Harman, 2012, Sec. 5.1)
    # Note: Fitness now returns total BV regardless of feasibility.
    # Feasibility is handled by the Deb selection operator.
    # ----------------------------------------------------------------------
    def fitness(chrom):
        total_bv, _, _, _ = evaluate_chrom(chrom)
        return total_bv

    # ----------------------------------------------------------------------
    # Deb-Style Tournament Selection (Deb, 2000)
    # ----------------------------------------------------------------------
    def deb_better(a_eval, b_eval):
        _, _, a_feas, a_viol = a_eval; _, _, b_feas, b_viol = b_eval
        if a_feas and not b_feas: return True
        if b_feas and not a_feas: return False
        if a_feas and b_feas: return a_eval[0] > b_eval[0] # Compare by BV
        return a_viol < b_viol # Compare by violation
    def deb_tournament_selection(population, k=3):
        contestants = random.sample(population, k=min(k, len(population))); best = contestants[0]
        best_eval = evaluate_chrom(best)
        for c in contestants[1:]:
            c_eval = evaluate_chrom(c)
            if deb_better(c_eval, best_eval): best, best_eval = c, c_eval
        return best

    # ----------------------------------------------------------------------
    # Crossover and Mutation Operators (Mitchell, 1998) - Unchanged
    # ----------------------------------------------------------------------
    def single_point_crossover(parent1, parent2):
        if n_requirements < 2: return parent1[:], parent2[:]
        point = random.randint(1, n_requirements - 1); child1 = parent1[:point] + parent2[point:]; child2 = parent2[:point] + parent1[point:]
        return child1, child2
    def mutate(chromosome, mutation_prob=mutation_prob):
        if random.random() < mutation_prob and n_requirements > 0:
            mutation_idx = random.randint(0, n_requirements - 1); chromosome[mutation_idx] = 1 - chromosome[mutation_idx]

    # ----------------------------------------------------------------------
    # Main GA Loop (modified to use new operators)
    # ----------------------------------------------------------------------
    for generation in range(max_generations):
        # --- Selection ---
        mating_pool = []
        for _ in range(population_size // 2):
            parent1 = deb_tournament_selection(population)
            parent2 = deb_tournament_selection(population)
            mating_pool.append((parent1, parent2))

        # --- Variation & Repair ---
        offspring = []
        for parent1, parent2 in mating_pool:
            if random.random() < crossover_prob: child1, child2 = single_point_crossover(parent1, parent2)
            else: child1, child2 = parent1[:], parent2[:]
            mutate(child1); mutate(child2)
            child1 = repair_value_aware(child1)
            child2 = repair_value_aware(child2)
            offspring.extend([child1, child2])

        # --- Replacement (using Deb's rules) ---
        combined_population = population + offspring
        def deb_key(ch):
            bv, _, feas, viol = evaluate_chrom(ch); return (0 if feas else 1, viol, -bv)
        combined_population.sort(key=deb_key)
        population = combined_population[:population_size]

    # Final best solution is the top of the population after the last sort
    best_solution = population[0]
    best_fitness, _, _, _ = evaluate_chrom(best_solution)
    selected_reqs = [all_req_ids[i] for i, val in enumerate(best_solution) if val]

    return selected_reqs, best_fitness

print("New GA logic has been refactored into a reusable function.")

New GA logic has been refactored into a reusable function.


In [ ]:
 # ===========================================
# Cell 3: Experiment Wrapper & Execution
# ===========================================

def run_all_experiments():
    filepath = DATA_FILE  # set by Cell 0

    if not os.path.exists(filepath):
        print(f"❌ File not found at: {filepath}")
        print("Check that D2.xlsx exists in the Data folder.")
        return None, None

    print(f"Loading data from '{filepath}'...")
    original_df     = pd.read_excel(filepath)
    TOTAL_EXEC_TIME = original_df.drop_duplicates(subset=['tc_id'])['tc_executiontime'].sum()
    print(f"Total possible execution time: {TOTAL_EXEC_TIME:.2f}")

    # Create data maps ONCE
    data_maps = {
        'req_to_tests':     original_df.groupby('us_id')['tc_id'].apply(set).to_dict(),
        'req_to_bv':        original_df.groupby('us_id')['us_businessvalue'].first().to_dict(),
        'test_to_time':     original_df.groupby('tc_id')['tc_executiontime'].first().to_dict(),
        'bval_to_reqs_map': original_df.groupby('us_businessvalue')['us_id'].apply(set).to_dict()
    }
    data_maps['all_req_ids'] = sorted(list(data_maps['req_to_tests'].keys()))

    all_version_results = {}
    for version_name, config in GA_CONFIGURATIONS.items():
        print("\n" + "="*80 + f"\n--- Running Experiment for Version: {version_name} ---")
        print(f"Parameters: {config}" + "\n" + "="*80)

        all_run_data = []
        for budget_pct in BUDGET_PERCENTAGES:
            budget_value = TOTAL_EXEC_TIME * (budget_pct / 100.0)
            print(f"\n--- Running for Budget: {budget_pct}% (Max Time: {budget_value:.2f}) ---")
            for run in range(NUM_RUNS_PER_BUDGET):
                print(f"  Run {run + 1}/{NUM_RUNS_PER_BUDGET}...", end='\r')
                selected_reqs, final_bv = run_ga_instance(budget_value, config, data_maps)
                all_run_data.append({
                    'budget_pct':           budget_pct,
                    'run':                  run + 1,
                    'total_business_value': final_bv,
                    'num_reqs_covered':     len(selected_reqs),
                    'selected_reqs':        selected_reqs
                })
            print(f"\n  -> Completed {NUM_RUNS_PER_BUDGET} runs.")

        all_version_results[version_name] = pd.DataFrame(all_run_data)

    return all_version_results, original_df

all_ga_results, original_df = run_all_experiments()

In [ ]:
# ========================================================
# Cell 4: Statistical Analysis and Final Report
# ========================================================

if 'all_ga_results' in locals() and all_ga_results:
    print("\n\n" + "="*60 + "\n--- Starting Final Analysis and Reporting ---")

    bcpso_filepath = BASELINE_FILE  # set by Cell 0

    if bcpso_filepath is None or not os.path.exists(bcpso_filepath):
        print(f"❌ Baseline file not found at: {bcpso_filepath}")
    else:
        bcpso_summary_df = pd.read_excel(bcpso_filepath, sheet_name='RTSGA_Full')
        print(f"✅ Loaded '{bcpso_filepath}' successfully.")

        for version_name, results_df in all_ga_results.items():
            print("\n" + "-"*60 + f"\nProcessing: {version_name}\n" + "-"*60)

            summary_stats_ga = results_df.groupby('budget_pct').agg(
                mean_reqs=('num_reqs_covered', 'mean'), median_reqs=('num_reqs_covered', 'median'),
                mean_bv=('total_business_value', 'mean'), median_bv=('total_business_value', 'median')
            ).reset_index()

            num_total_reqs = original_df['us_id'].nunique()
            summary_stats_ga['Req Coverage %'] = (summary_stats_ga['mean_reqs'] / num_total_reqs) * 100

            print(f"\n--- {version_name} Final Summary Table ---")
            print(summary_stats_ga.to_string())

            # --- Individual GA Plots ---
            fig_indiv, axes_indiv = plt.subplots(2, 2, figsize=(20, 15))
            fig_indiv.suptitle(f'RTS-GA Individual Analysis: {version_name}', fontsize=18)
            sns.boxplot(ax=axes_indiv[0, 0], x='budget_pct', y='total_business_value', data=results_df)
            axes_indiv[0, 0].set_title('Distribution of Total BV (30 Runs)')
            axes_indiv[0, 0].tick_params(axis='x', rotation=45)
            axes_indiv[0, 1].plot(summary_stats_ga['budget_pct'], summary_stats_ga['Req Coverage %'], marker='o')
            axes_indiv[0, 1].set_title('Mean Req Coverage % vs. Budget')
            sns.barplot(ax=axes_indiv[1, 0], x='budget_pct', y='median_bv', data=summary_stats_ga, color='purple', alpha=0.8)
            axes_indiv[1, 0].set_title('Median Business Value vs. Budget')
            axes_indiv[1, 0].tick_params(axis='x', rotation=45)
            sns.barplot(ax=axes_indiv[1, 1], x='budget_pct', y='median_reqs', data=summary_stats_ga, color='orange', alpha=0.8)
            axes_indiv[1, 1].set_title('Median # of Requirements Covered vs. Budget')
            for ax in axes_indiv.flat:
                ax.set_xlabel('Time Budget (%)')
                ax.grid(True, linestyle='--', alpha=0.7)
            plt.tight_layout(rect=[0, 0, 1, 0.96])
            plt.show()

            # --- Comparative Plots ---
            fig_comp, axes_comp = plt.subplots(2, 2, figsize=(20, 15))
            fig_comp.suptitle(f'Comparative Analysis: {version_name} vs. RTSGA Baseline', fontsize=18)

            axes_comp[0, 0].plot(summary_stats_ga['budget_pct'], summary_stats_ga['Req Coverage %'], marker='o', label=f'{version_name}')
            axes_comp[0, 0].plot(bcpso_summary_df['cycle'], bcpso_summary_df['req_cov_mean'], marker='x', label='RTSGA Baseline')
            axes_comp[0, 0].fill_between(bcpso_summary_df['cycle'],
                                          bcpso_summary_df['req_cov_mean'] - bcpso_summary_df['req_cov_std'],
                                          bcpso_summary_df['req_cov_mean'] + bcpso_summary_df['req_cov_std'],
                                          alpha=0.15, label='Baseline ±1 std')
            axes_comp[0, 0].set_title('Req Coverage %: GA vs. RTSGA Baseline')
            axes_comp[0, 0].legend(); axes_comp[0, 0].set_ylabel('Coverage (%)')

            axes_comp[0, 1].plot(summary_stats_ga['budget_pct'], summary_stats_ga['Req Coverage %'], marker='o', label=f'{version_name}')
            axes_comp[0, 1].plot(bcpso_summary_df['cycle'], bcpso_summary_df['req_cov_median'], marker='x', label='RTSGA Baseline (Median)')
            axes_comp[0, 1].set_title('Req Coverage % (Median): GA vs. RTSGA Baseline')
            axes_comp[0, 1].legend(); axes_comp[0, 1].set_ylabel('Coverage (%)')

            axes_comp[1, 0].plot(summary_stats_ga['budget_pct'], summary_stats_ga['mean_bv'], marker='o', label=f'{version_name}')
            axes_comp[1, 0].plot(bcpso_summary_df['cycle'], bcpso_summary_df['bv_mean'], marker='x', label='RTSGA Baseline')
            axes_comp[1, 0].fill_between(bcpso_summary_df['cycle'],
                                          bcpso_summary_df['bv_mean'] - bcpso_summary_df['bv_std'],
                                          bcpso_summary_df['bv_mean'] + bcpso_summary_df['bv_std'],
                                          alpha=0.15, label='Baseline ±1 std')
            axes_comp[1, 0].set_title('Mean BV: GA vs. RTSGA Baseline')
            axes_comp[1, 0].legend(); axes_comp[1, 0].set_ylabel('Business Value')

            axes_comp[1, 1].plot(summary_stats_ga['budget_pct'], summary_stats_ga['median_bv'], marker='o', label=f'{version_name}')
            axes_comp[1, 1].plot(bcpso_summary_df['cycle'], bcpso_summary_df['bv_median'], marker='x', label='RTSGA Baseline (Median)')
            axes_comp[1, 1].set_title('Median BV: GA vs. RTSGA Baseline')
            axes_comp[1, 1].legend(); axes_comp[1, 1].set_ylabel('Business Value')

            for ax in axes_comp.flat:
                ax.set_xlabel('Cycle / Budget %')
                ax.grid(True, linestyle='--', alpha=0.7)
                ax.tick_params(axis='x', rotation=45)
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()

            output_excel_filename = os.path.join(OUTPUT_DIR, f"Comprehensive_Report_{version_name}.xlsx")
            print(f"\nGenerating report: '{output_excel_filename}'")
            img_buffer_indiv = BytesIO(); fig_indiv.savefig(img_buffer_indiv, format='png'); img_buffer_indiv.seek(0)
            img_buffer_comp  = BytesIO(); fig_comp.savefig(img_buffer_comp,  format='png'); img_buffer_comp.seek(0)
            with pd.ExcelWriter(output_excel_filename, engine='openpyxl') as writer:
                results_df.to_excel(writer,        sheet_name='RTS-GA Raw Runs',    index=False)
                summary_stats_ga.to_excel(writer,  sheet_name='RTS-GA Summary',     index=False)
                bcpso_summary_df.to_excel(writer,  sheet_name='RTSGA Baseline',     index=False)
                ws = writer.book.create_sheet(title="Analysis Plots")
                ws.add_image(OpenpyxlImage(img_buffer_indiv), 'A1')
                ws.add_image(OpenpyxlImage(img_buffer_comp),  'A80')
            print(f"✅ Report saved: '{output_excel_filename}'")

            # In Colab, also offer download of the report
            if IN_COLAB:
                try:
                    from google.colab import files as _cf
                    _cf.download(output_excel_filename)
                except Exception:
                    pass

else:
    print("\nNo RTS-GA data collected.")